# 📊 Conciliação em Massa — Motor v5

Cruza **TODOS** os pagamentos da planilha de pendências com os títulos em aberto.
Solver MILP multi-valor (Saldo, −IR, −ISS, Líquido). Gera Excel com resultado.

> Requer duas planilhas: **Pendências** (pagamentos) + **Títulos em aberto**

In [ ]:
!pip install pandas openpyxl scipy -q

In [ ]:
import pandas as pd, numpy as np, time, io, re
from difflib import SequenceMatcher
from scipy.optimize import milp, LinearConstraint, Bounds
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

TOLERANCIA = 0.01; MAX_NOTAS = 30; TIMEOUT = 20
STATUS_RESOLVIDO = {'BAIXADO','BAIXADO ','baixado'}
ABAS_IGNORAR = {'classificação','PORTAIS','PG INTERMUNICIPAIS','ADIANTAMENTOS'}

def norm(t):
    if pd.isna(t): return ''
    t = str(t).upper().strip()
    for p in ['PREFEITURA MUNICIPAL DE ','PREFEITURA MUNICIPAL ','FUNDO MUNICIPAL DE SAUDE DE ','FUNDO MUNICIPAL DE SAUDE ','FUNDO DE SAUDE DE ','FUNDO DE SAUDE ','MUNICIPIO DE ','SECRETARIA DE ','SECRETARIA MUNICIPAL DE ','SEC ','PM ','PREF ','PREF. ','FMS ','FME ']:
        t = t.replace(p,' ')
    return re.sub(r'\s+',' ',t).strip()

def sim(a,b): return SequenceMatcher(None,a,b).ratio()
def centavos(v):
    try: return round(float(v)*100)
    except: return 0

def gerar_opcoes(row):
    s=round(float(row.get('SALDO',0) or 0),2); ir=round(float(row.get('IR',0) or 0),2); iss=round(float(row.get('ISS',0) or 0),2)
    ops=[('saldo',s)]
    if ir!=0: ops.append(('saldo_ir',round(s+ir,2)))
    if iss!=0: ops.append(('saldo_iss',round(s+iss,2)))
    if ir!=0 and iss!=0: ops.append(('liquido',round(s+ir+iss,2)))
    seen,d=set(),[]
    for n,v in ops:
        if v>0 and v not in seen: seen.add(v); d.append((n,v))
    return d

def solver_milp(df_n, alvo):
    keys,vals=[],[]
    for idx,row in df_n.iterrows():
        for var,val in gerar_opcoes(row):
            if 0<val<=alvo+TOLERANCIA: keys.append((idx,var)); vals.append(val)
    n=len(vals)
    if n==0: return 'nenhum',[]
    va=np.array(vals)
    ng={}
    for i,(idx,var) in enumerate(keys): ng.setdefault(idx,[]).append(i)
    cs=[LinearConstraint(va.reshape(1,-1),lb=alvo-TOLERANCIA,ub=alvo+TOLERANCIA),LinearConstraint(np.ones((1,n)),lb=0,ub=MAX_NOTAS)]
    for indices in ng.values():
        if len(indices)>1:
            r=np.zeros(n)
            for i in indices: r[i]=1
            cs.append(LinearConstraint(r.reshape(1,-1),lb=0,ub=1))
    res=milp(np.ones(n),constraints=cs,integrality=np.ones(n),bounds=Bounds(lb=np.zeros(n),ub=np.ones(n)))
    if res.success and res.x is not None:
        xi=np.round(res.x).astype(int)
        sel=[(keys[i][0],keys[i][1],vals[i]) for i in range(n) if xi[i]==1]
        return 'unico',sel
    return 'nenhum',[]

def selecionar(pag, titulos):
    mun=pag.get('municipio_norm','')
    if mun:
        ex=titulos[titulos['cidade_norm']==mun]
        if len(ex)>0: return ex,f'Município Exato ({pag.get("municipio","")})'
    if mun and len(mun)>=3:
        sc=titulos['cidade_norm'].apply(lambda c:sim(mun,c) if c else 0)
        m=sc>=0.70
        if m.any():
            best=sc[m].max(); cidade=titulos.loc[sc==best,'CIDADE'].iloc[0]
            return titulos[titulos['CIDADE']==cidade],f'Município Fuzzy ({cidade},{best:.0%})'
    hist=pag.get('historico_norm','')
    if hist and len(hist)>=5:
        sc=titulos['nome_norm'].apply(lambda n:sim(hist,n) if n else 0)
        m=sc>=0.65
        if m.any():
            best=sc[m].max(); cidade=titulos.loc[sc==best,'CIDADE'].iloc[0]
            return titulos[titulos['CIDADE']==cidade],f'Histórico Fuzzy ({best:.0%})'
    return pd.DataFrame(),'Sem candidatos'

VL={'saldo':'Saldo','saldo_ir':'Saldo−IR','saldo_iss':'Saldo−ISS','liquido':'Líquido'}
CV={'saldo':{'SALDO'},'saldo_ir':{'SALDO','IR'},'saldo_iss':{'SALDO','ISS'},'liquido':{'SALDO','IR','ISS'}}

print('✅ Motor em massa carregado')


## 📂 Upload das planilhas

In [ ]:
from google.colab import files
print('📂 Upload da PLANILHA DE PENDÊNCIAS:')
up1 = files.upload()
pend_name = list(up1.keys())[0]
pend_bytes = up1[pend_name]

print('\n📂 Upload dos TÍTULOS EM ABERTO:')
up2 = files.upload()
tit_name = list(up2.keys())[0]
tit_bytes = up2[tit_name]

# Ler títulos
titulos = pd.read_excel(io.BytesIO(tit_bytes))
titulos.columns = titulos.columns.str.strip()
for c in ['SALDO','BRUTO','CORRETO','IR','ISS']:
    if c in titulos.columns: titulos[c]=pd.to_numeric(titulos[c].astype(str).str.replace(',','.'),errors='coerce')
for c in ['SALDO','IR','ISS']:
    if c not in titulos.columns: titulos[c]=0.0
    titulos[c]=titulos[c].fillna(0.0)
titulos['VENCIMENTO']=pd.to_datetime(titulos.get('VENCIMENTO',pd.NaT),errors='coerce')
titulos['CIDADE']=titulos['CIDADE'].str.strip().str.upper() if 'CIDADE' in titulos.columns else ''
titulos['cidade_norm']=titulos['CIDADE'].apply(norm)
titulos['nome_norm']=titulos['NOME CLIENTE'].apply(norm) if 'NOME CLIENTE' in titulos.columns else ''
titulos['idx_titulo']=range(len(titulos))
titulos=titulos[(titulos['SALDO'].notna())&(titulos['SALDO']>0)].copy()
print(f'\n✅ {len(titulos):,} títulos carregados')

# Listar abas
xl = pd.ExcelFile(io.BytesIO(pend_bytes))
abas = [s for s in xl.sheet_names if s not in ABAS_IGNORAR]
print(f'\nAbas disponíveis: {abas}')


## ⚙️ Selecionar aba e executar

In [ ]:
#@title Selecione a aba { run: "auto" }
ABA = 'MAR 26' #@param {type:"string"}

# Ler pendências
raw=pd.read_excel(io.BytesIO(pend_bytes),sheet_name=ABA,dtype=str,header=None)
hr=None
for i,row in raw.iterrows():
    vs=[str(v).strip().upper() for v in row if not pd.isna(v)]
    if any(v in ('VALOR','DT LANÇAMENTO','HISTORICO','HISTÓRICO') for v in vs): hr=i; break
assert hr is not None, 'Cabeçalho não encontrado'
df_r=raw.iloc[hr:].copy(); df_r.columns=df_r.iloc[0].str.strip(); df_r=df_r.iloc[1:].reset_index(drop=True)
cm={'data':['DT LANÇAMENTO','DATA'],'banco':['BANCO'],'status':['STATUS'],'valor':['VALOR','VALOR PAGO','VL PAGO'],'historico':['HISTÓRICO','HISTORICO'],'municipio':['MUNICIPIO','MUNICÍPIO','CIDADE']}
cd={c.strip().upper():c for c in df_r.columns if not pd.isna(c)}
mp={}
for campo,ops in cm.items(): mp[campo]=next((cd[o.upper()] for o in ops if o.upper() in cd),None)
banco=pd.DataFrame()
banco['data']=pd.to_datetime(df_r[mp['data']],errors='coerce') if mp['data'] else pd.NaT
banco['banco']=df_r[mp['banco']].str.strip() if mp['banco'] else ''
banco['status']=df_r[mp['status']].str.strip() if mp['status'] else ''
banco['valor']=pd.to_numeric(df_r[mp['valor']].astype(str).str.replace(',','.'),errors='coerce') if mp['valor'] else 0
banco['historico']=df_r[mp['historico']].str.strip() if mp['historico'] else ''
banco['municipio']=df_r[mp['municipio']].str.strip().str.upper() if mp['municipio'] else ''
banco['municipio_norm']=banco['municipio'].apply(norm)
banco['historico_norm']=banco['historico'].apply(norm)
banco['valor_cent']=banco['valor'].apply(centavos)
banco=banco.dropna(subset=['valor']); banco=banco[banco['valor']>0]
mb=banco['status'].str.upper().str.strip().isin(STATUS_RESOLVIDO)
n_baix=mb.sum(); banco=banco[~mb].reset_index(drop=True)
print(f'✅ {len(banco)} pagamentos pendentes ({n_baix} baixados ignorados)')
print(f'   Total: R$ {banco["valor"].sum():,.2f}')


In [ ]:
# ══════════════════════════════════════════════════
# EXECUTAR CONCILIAÇÃO EM MASSA
# ══════════════════════════════════════════════════
resultados=[]; titulos_usados=set()
banco['chave_grupo']=banco['municipio'].fillna('')+'|'+banco['data'].dt.strftime('%Y-%m-%d').fillna('')
grupos=list(banco.groupby('chave_grupo',sort=False))
total=len(banco); proc=0
t0_total=time.time()

for gi,(chave,grupo) in enumerate(grupos):
    pags=grupo.sort_values('valor',ascending=False)
    ref=pags.iloc[0]
    cands,fase=selecionar(ref,titulos)
    if len(cands)==0:
        for _,p in pags.iterrows():
            resultados.append({'pag':p,'status':'SEM_CANDIDATOS','fase':fase,'solucao':[],'obs':'Município não encontrado'})
            proc+=1
        continue
    cl=cands[~cands['idx_titulo'].isin(titulos_usados)].copy()
    if len(cl)==0:
        for _,p in pags.iterrows():
            resultados.append({'pag':p,'status':'SEM_CANDIDATOS','fase':fase,'solucao':[],'obs':'Títulos esgotados'})
            proc+=1
        continue
    ug=set()
    for _,p in pags.iterrows():
        al=round(float(p['valor']),2)
        df_livre=cl[~cl['idx_titulo'].isin(ug)]
        if len(df_livre)==0:
            resultados.append({'pag':p,'status':'SEM_CANDIDATOS','fase':fase,'solucao':[],'obs':'Esgotados'})
        else:
            st,sol=solver_milp(df_livre,al)
            if st=='unico':
                resultados.append({'pag':p,'status':'unico','fase':fase,'solucao':sol,'obs':''})
                for idx_t,_,_ in sol: ug.add(idx_t); titulos_usados.add(idx_t)
            else:
                resultados.append({'pag':p,'status':st,'fase':fase,'solucao':[],'obs':'Sem combinação'})
        proc+=1
        if proc%20==0: print(f'  {proc}/{total} processados...')

elapsed=time.time()-t0_total
n_ok=sum(1 for r in resultados if r['status']=='unico')
n_amb=sum(1 for r in resultados if r['status']=='ambiguo')
n_sem=len(resultados)-n_ok-n_amb
taxa=n_ok/len(resultados)*100 if resultados else 0

print(f'\n{"="*70}')
print(f'  RESULTADO — {ABA}')
print(f'{"="*70}')
print(f'  ✅ Conciliados: {n_ok}')
print(f'  ⚠️  Ambíguos:   {n_amb}')
print(f'  ❌ Sem match:   {n_sem}')
print(f'  📊 Taxa:        {taxa:.0f}%')
print(f'  ⏱️  Tempo:       {elapsed:.0f}s')
print(f'{"="*70}')


## 📥 Gerar Excel

In [ ]:
wb=Workbook(); wb.remove(wb.active)
FT='Arial'; VBG='C6EFCE'; VFT='276221'; NBG='F7F7F7'; NFT='666666'
ABG='FFEB9C'; AFT='9C6500'
def bd():
    s=Side(style='thin',color='C0C0C0'); return Border(left=s,right=s,top=s,bottom=s)
def cl(ws,r,c,v=None,bold=False,color='000000',bg=None,sz=9,ha='left'):
    cell=ws.cell(row=r,column=c)
    if v is not None: cell.value=v
    cell.font=Font(name=FT,bold=bold,color=color,size=sz)
    cell.alignment=Alignment(horizontal=ha,vertical='center')
    cell.border=bd()
    if bg: cell.fill=PatternFill('solid',start_color=bg)
    return cell

ok=[r for r in resultados if r['status']=='unico']
nok=[r for r in resultados if r['status']!='unico']
tit_map={int(row['idx_titulo']):row for _,row in titulos.iterrows()}
COLS=[('NF',10),('DOC',14),('VENCIMENTO',13),('SALDO',15),('IR',11),('ISS',11),('CORRETO',15),('NOME CLIENTE',38),('UF',6),('TIPO',14)]

ws=wb.create_sheet('CONCILIAÇÃO')
ws.sheet_view.showGridLines=False
for i,(_,w) in enumerate(COLS,1): ws.column_dimensions[get_column_letter(i)].width=w
ws.merge_cells('A1:J1'); ws['A1'].value=f'CONCILIAÇÃO EM MASSA — {ABA}'
ws['A1'].font=Font(name=FT,bold=True,size=13,color='FFFFFF'); ws['A1'].fill=PatternFill('solid',start_color='1F3864')
ws['A1'].alignment=Alignment(horizontal='center',vertical='center')
row=3
for res in ok:
    pag=res['pag']; sol=res['solucao']
    for c,h in enumerate(['DATA','BANCO','VALOR','HISTÓRICO','MUNICÍPIO','','','','',''],1): cl(ws,row,c,h,bold=True,color='FFFFFF',bg='1F3864',ha='center',sz=8)
    row+=1
    vs=[pag['data'].date() if pd.notna(pag.get('data')) else '',pag.get('banco',''),pag['valor'],pag.get('historico',''),pag.get('municipio',''),'','','','','']
    for c,v in enumerate(vs,1):
        cell=cl(ws,row,c,v,bold=True,color='FFFFFF',bg='2E4057',ha='center' if c<=3 else 'left',sz=10)
        if c==3: cell.number_format='#,##0.00'
    row+=1
    for c,(campo,_) in enumerate(COLS,1): cl(ws,row,c,campo,bold=True,color='FFFFFF',bg='2E75B6',ha='center',sz=8)
    row+=1
    for idx_t,variante,val_c in sol:
        tit=tit_map.get(idx_t); 
        if not tit: continue
        cv=CV.get(variante,set())
        for c,(campo,_) in enumerate(COLS,1):
            if campo=='TIPO':
                cl(ws,row,c,VL.get(variante,variante),bold=True,color=AFT,bg=ABG,ha='center')
            else:
                val=tit.get(campo,'')
                if campo=='VENCIMENTO' and pd.notna(val):
                    try: val=val.date()
                    except: pass
                if campo in cv:
                    cell=cl(ws,row,c,val,bold=True,color=VFT,bg=VBG,ha='right' if campo in ('SALDO','IR','ISS','CORRETO') else 'center')
                else:
                    cell=cl(ws,row,c,val,color=NFT,bg=NBG,ha='right' if campo in ('SALDO','IR','ISS','CORRETO') else 'center')
                if campo in ('SALDO','IR','ISS','CORRETO'): cell.number_format='#,##0.00'
        row+=1
    for c in range(1,11): ws.cell(row=row,column=c).fill=PatternFill('solid',start_color='F0F0F0')
    row+=1

ws2=wb.create_sheet('NÃO RESOLVIDOS')
for c,h in enumerate(['Data','Banco','Valor','Histórico','Município','Fase','Motivo'],1): cl(ws2,1,c,h,bold=True,color='FFFFFF',bg='9C0006',ha='center')
for i,res in enumerate(nok,2):
    pag=res['pag']
    for c,v in enumerate([pag['data'].date() if pd.notna(pag.get('data')) else '',pag.get('banco',''),pag['valor'],pag.get('historico',''),pag.get('municipio',''),res.get('fase',''),res.get('obs','')],1):
        cell=cl(ws2,i,c,v,color='9C0006',bg='FFC7CE')
        if c==3: cell.number_format='#,##0.00'
for l,w in zip('ABCDEFG',[12,8,14,40,20,22,40]): ws2.column_dimensions[l].width=w

nome_saida=f'conciliacao_massa_{ABA.replace(" ","_")}.xlsx'
wb.save(nome_saida)
files.download(nome_saida)
print(f'\n✅ Excel salvo e baixado: {nome_saida}')
print(f'   {len(ok)} conciliados | {len(nok)} não resolvidos')
